In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
!unzip /content/drive/MyDrive/"My First Project.v14i.coco.zip" -d /content/drive/MyDrive/weed_dataset

Archive:  /content/drive/MyDrive/My First Project.v14i.coco.zip
  inflating: /content/drive/MyDrive/weed_dataset/README.dataset.txt  
  inflating: /content/drive/MyDrive/weed_dataset/README.roboflow.txt  
   creating: /content/drive/MyDrive/weed_dataset/test/
 extracting: /content/drive/MyDrive/weed_dataset/test/0001_jpeg.rf.8aa88c044db8631103fc5f47f59b22cf.jpg  
 extracting: /content/drive/MyDrive/weed_dataset/test/0003_jpeg.rf.d803dd550d9fd12004b0fe09e33935d9.jpg  
 extracting: /content/drive/MyDrive/weed_dataset/test/0004_jpeg.rf.89b9b142ac8b202ec3b80b07c78abab4.jpg  
 extracting: /content/drive/MyDrive/weed_dataset/test/0009_jpeg.rf.35984c3c49186c09ef47adf3e6848d24.jpg  
 extracting: /content/drive/MyDrive/weed_dataset/test/0010_jpeg.rf.04c6016add042909858899978e0f32ab.jpg  
 extracting: /content/drive/MyDrive/weed_dataset/test/0011_jpeg.rf.1872b1ca5c43bac27ee05c4e7b4c7fde.jpg  
 extracting: /content/drive/MyDrive/weed_dataset/test/0012_jpeg.rf.1755aad1a90506d7e82f13762086d37b.jpg 

In [8]:
!ls /content/drive/MyDrive/weed_dataset

README.dataset.txt  README.roboflow.txt  test  train  valid


In [10]:
!ls /content/drive/MyDrive/weed_dataset/train

101_jpeg.rf.a04f6a164b9182c77c1a8f7e400ec565.jpg
102_jpeg.rf.16de49c622227514b59654710c5fffed.jpg
103_jpeg.rf.b0fc654342ceb1d2be6ed8c7d78d18f3.jpg
104_jpeg.rf.135dc8d3abf294075d90db4231dbc734.jpg
105_jpeg.rf.70fe39982b0814c4ffdbe6cc19928ca0.jpg
106_jpeg.rf.608509e3b0d4a861cc5cf61f8d9a77c2.jpg
107_jpeg.rf.8a9e8ad2af210710aea06a963ee1db65.jpg
108_jpeg.rf.c66c446b6a3e9173742c5d7690db57f5.jpg
109_jpeg.rf.b35f398784ea56ee1f3b4002cdc14b46.jpg
110_jpeg.rf.198564c0d97e72fa11b0317a5376f3f8.jpg
111_jpeg.rf.676e6b66286a62ac04ea8265a21547dc.jpg
112_jpeg.rf.dc4b0ffaf8c84a6393d32cd5ff120995.jpg
113_jpeg.rf.24c8ae431e1c5a1a66a955f43423c3be.jpg
114_jpeg.rf.da554e24cb121d028bb52042b74f78a3.jpg
115_jpeg.rf.6ac188ad6e821b7c5c1aafa8289022a7.jpg
116_jpeg.rf.63058843a5ca7401c9bbefb726f61452.jpg
117_jpeg.rf.1eeba14cdc168a51b8dedf5d6b659cbe.jpg
118_jpeg.rf.803cd98f751942a90b65110295a73992.jpg
119_jpeg.rf.36caa0bec68c931b428e276d16967a9c.jpg
120_jpeg.rf.9b0513cb6a793eab0a7daf67c71efc11.jpg
121_jpeg.rf.34ebc5f2

In [11]:
!pip install pycocotools

In [4]:
import torch
import torchvision
import os
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# =========================
# DATASET CLASS (ROBUST)
# =========================
class WeedPlantDataset(Dataset):
    def __init__(self, img_dir, ann_file):
        self.img_dir = img_dir
        self.coco = COCO(ann_file)
        self.ids = list(self.coco.imgs.keys())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        while True:  # retry mechanism
            img_id = self.ids[idx]
            img_info = self.coco.loadImgs(img_id)[0]

            path = img_info['file_name']
            img_path = os.path.join(self.img_dir, path)

            if not os.path.exists(img_path):
                idx = (idx + 1) % len(self)
                continue

            img = Image.open(img_path).convert("RGB")
            img = torchvision.transforms.functional.to_tensor(img)

            ann_ids = self.coco.getAnnIds(imgIds=img_id)
            anns = self.coco.loadAnns(ann_ids)

            boxes, labels, masks = [], [], []

            for ann in anns:
                segmentation = ann.get("segmentation", [])

                # skip invalid segmentation
                if not isinstance(segmentation, list) or len(segmentation) == 0:
                    continue

                try:
                    rles = coco_mask.frPyObjects(
                        segmentation,
                        img_info['height'],
                        img_info['width']
                    )
                    rle = coco_mask.merge(rles)
                    mask = coco_mask.decode(rle)
                except:
                    continue

                x, y, w, h = ann['bbox']
                boxes.append([x, y, x + w, y + h])
                labels.append(ann['category_id'])
                masks.append(mask)

            # skip images with no valid objects
            if len(boxes) == 0:
                idx = (idx + 1) % len(self)
                continue

            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            masks = torch.as_tensor(np.array(masks), dtype=torch.uint8)

            target = {
                "boxes": boxes,
                "labels": labels,
                "masks": masks,
                "image_id": torch.tensor([img_id])
            }

            return img, target

# =========================
# PATHS
# =========================
train_img = "/content/drive/MyDrive/weed_dataset/train"
train_ann = "/content/drive/MyDrive/weed_dataset/train/_annotations.coco.json"

# =========================
# DATALOADER
# =========================
def collate_fn(batch):
    return tuple(zip(*batch))

train_dataset = WeedPlantDataset(train_img, train_ann)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

# =========================
# MODEL
# =========================
num_classes = 3  # background + plant + weed

model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

# Replace box head
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
    in_features, num_classes
)

# Replace mask head
in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
model.roi_heads.mask_predictor = torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(
    in_features_mask, 256, num_classes
)

model.to(device)

# =========================
# OPTIMIZER
# =========================
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# =========================
# MIXED PRECISION
# =========================
scaler = torch.amp.GradScaler("cuda")

# =========================
# TRAINING LOOP
# =========================
num_epochs = 25

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for imgs, targets in train_loader:

        imgs = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            loss_dict = model(imgs, targets)
            losses = sum(loss for loss in loss_dict.values())

        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += losses.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# =========================
# SAVE MODEL
# =========================
torch.save(model.state_dict(), "/content/drive/MyDrive/maskrcnn_final.pth")

print("✅ Training complete & model saved!")

Using: cuda
loading annotations into memory...
Done (t=0.08s)
creating index...
index created!
Epoch 1, Loss: 395.5070
Epoch 2, Loss: 317.3143
Epoch 3, Loss: 281.0552
Epoch 4, Loss: 252.2038
Epoch 5, Loss: 229.1775
Epoch 6, Loss: 205.5842
Epoch 7, Loss: 185.0187
Epoch 8, Loss: 168.1195
Epoch 9, Loss: 155.8405
Epoch 10, Loss: 145.4213
Epoch 11, Loss: 131.7970
Epoch 12, Loss: 123.6637
Epoch 13, Loss: 115.5367
Epoch 14, Loss: 115.6751
Epoch 15, Loss: 110.1785
Epoch 16, Loss: 107.3893
Epoch 17, Loss: 99.8730
Epoch 18, Loss: 96.1020
Epoch 19, Loss: 99.5991
Epoch 20, Loss: 97.1334
Epoch 21, Loss: 95.0354
Epoch 22, Loss: 94.1520
Epoch 23, Loss: 82.4470
Epoch 24, Loss: 78.2731
Epoch 25, Loss: 77.3111
✅ Training complete & model saved!


In [5]:
import torch
from pycocotools.cocoeval import COCOeval
from pycocotools.coco import COCO
import numpy as np

def evaluate(model, data_loader, device, ann_file):
    model.eval()

    coco_gt = COCO(ann_file)
    results = []

    with torch.no_grad():
        for imgs, targets in data_loader:
            imgs = [img.to(device) for img in imgs]

            outputs = model(imgs)

            for i, output in enumerate(outputs):
                img_id = targets[i]["image_id"].item()

                boxes = output["boxes"].cpu().numpy()
                scores = output["scores"].cpu().numpy()
                labels = output["labels"].cpu().numpy()
                masks = output["masks"].cpu().numpy()

                for j in range(len(boxes)):
                    x1, y1, x2, y2 = boxes[j]
                    w = x2 - x1
                    h = y2 - y1

                    results.append({
                        "image_id": img_id,
                        "category_id": int(labels[j]),
                        "bbox": [float(x1), float(y1), float(w), float(h)],
                        "score": float(scores[j])
                    })

    # Save results
    import json
    with open("results.json", "w") as f:
        json.dump(results, f)

    coco_dt = coco_gt.loadRes("results.json")

    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    return coco_eval

In [7]:
# =========================
# PATHS
# =========================
train_img = "/content/drive/MyDrive/weed_dataset/train"
train_ann = "/content/drive/MyDrive/weed_dataset/train/_annotations.coco.json"

val_img = "/content/drive/MyDrive/weed_dataset/valid"
val_ann = "/content/drive/MyDrive/weed_dataset/valid/_annotations.coco.json"

# =========================
# DATASETS
# =========================
train_dataset = WeedPlantDataset(train_img, train_ann)
val_dataset = WeedPlantDataset(val_img, val_ann)  # 🔥 THIS WAS MISSING

# =========================
# DATALOADER
# =========================
def collate_fn(batch):
    return tuple(zip(*batch))

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

# =========================
# EVALUATION
# =========================
coco_eval = evaluate(model, val_loader, device, val_ann)

loading annotations into memory...
Done (t=0.08s)
creating index...
index created!
loading annotations into memory...
Done (t=0.59s)
creating index...
index created!
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.12s).
Accumulating evaluation results...
DONE (t=0.03s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.436
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.575
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.460
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.443
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | m